In [2]:
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
#from geomloss import SamplesLoss
from modules import UNet, UNet2
from data import get_dataloader
from rectified_flow import RectifiedFlow
from utils import create_save_dir, log_results, plot_generated_images, plot_wasserstein_gamma
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def load_model(model_path, device, c_in, img_size):
    # Load the UNet model and its weights
    model = UNet2(device=device, img_size=img_size, c_in=c_in, c_out=c_in).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()  # Set to evaluation mode
    return model

def generate_samples(rect_flow: RectifiedFlow, model,  save_dir, epoch, time_steps=200, time_grid  = 'geometric'):
    
    # Generate samples and the expected norm suqared of d model(x_t, t)/ dt on a fine grid of points
    generated_samples = rect_flow.sample(model, n=1000, time_steps=time_steps, time_grid=time_grid)
    plot_generated_images(generated_samples, epoch = epoch, time_steps=time_steps, save_dir=save_dir, time_grid= time_grid)
    return generated_samples

    
    


In [4]:
data = 'MNIST' # MNIST or CIFAR 10

base_save_dir = "./saved/{}".format(data)
save_dir = create_save_dir(base_save_dir, selected_classes=None, selected_attributes=None)
save_dir

'./saved/MNIST/classes_all'

In [5]:
if data == 'MNIST':
    c_in = 1
    img_size = 32

if data == 'CIFAR10':
    c_in = 3
    img_size = 32
epoch = 30

model_path = os.path.join(save_dir, "models-3", f"unet_epoch_{epoch}.pth")
print(model_path)
model = load_model(model_path, device, c_in, img_size).to(device)
rect_flow = RectifiedFlow(device, img_size, c_in)
# Define the range of disc_steps to test
#disc_steps_list = list(range(1, 31, 1))
gen_linear = generate_samples(rect_flow, model, save_dir, epoch, time_steps = 100, time_grid = 'linear')
gen_geometric = generate_samples(rect_flow, model, save_dir, epoch, time_steps = 100, time_grid = 'geometric')


./saved/MNIST/classes_all/models-3/unet_epoch_30.pth
Sampling 1000 new images in batches of 128


 12%|█▎        | 1/8 [00:12<01:25, 12.23s/it]


KeyboardInterrupt: 

In [6]:
TIME_STEPS = 100
epoch = 30
generation_save_dir = os.path.join(save_dir, "generated_samples")
os.makedirs(generation_save_dir, exist_ok=True) # Ensure the directory exists

# Define filenames including relevant parameters
linear_filename = f"epoch_{epoch}_steps_{TIME_STEPS}_linear.pt"
geometric_filename = f"epoch_{epoch}_steps_{TIME_STEPS}_geometric.pt"

# Save the generated tensors
linear_save_path = os.path.join(generation_save_dir, linear_filename)
geometric_save_path = os.path.join(generation_save_dir, geometric_filename)

# torch.save(gen_linear.cpu(), linear_save_path)
# torch.save(gen_geometric.cpu(), geometric_save_path)

print(f"\nSaved Linear samples to: {linear_save_path}")
print(f"Saved Geometric samples to: {geometric_save_path}")





def load_generated_samples(path, device):
    """Loads a saved tensor and moves it to the specified device."""
    # Loading to CPU first is safer for cross-device loading
    loaded_tensor = torch.load(path, map_location=torch.device('cpu'))
    return loaded_tensor.to(device)

print("\nDemonstrating loading the saved files...")
loaded_linear = load_generated_samples(linear_save_path, device)
loaded_geometric = load_generated_samples(geometric_save_path, device)
print(f"Successfully loaded  samples. Shape: {loaded_linear.shape}")



Saved Linear samples to: ./saved/MNIST/classes_all/generated_samples/epoch_30_steps_100_linear.pt
Saved Geometric samples to: ./saved/MNIST/classes_all/generated_samples/epoch_30_steps_100_geometric.pt

Demonstrating loading the saved files...
Successfully loaded  samples. Shape: torch.Size([1000, 1, 32, 32])


In [22]:
pip install torch-fidelity

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import numpy as np
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchmetrics.image.fid import FrechetInceptionDistance
from tqdm import tqdm # For progress bar

# --- Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if data == 'CIFAR10':
    IMG_SIZE = 32
    FID_BATCH_SIZE = 128
    DATA_ROOT = './cifar10_data' # Directory to save downloaded CIFAR-10 data
if data == "MNIST":
    

# FID is calculated against the entire training set for comparison
# Standard CIFAR-10 normalization for FID calculation
CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD = [0.2023, 0.1994, 0.2010]
DATA_RANGE = (-1., 1.) # Assumed range of your UNet output and dataloader output

# --- 1. CIFAR-10 Dataloader Implementation ---

def get_cifar10_train_dataloader(batch_size, img_size):
    """Loads the CIFAR-10 training set for use in FID calculation."""
    
    # We use a simple transform that ensures the data is in the expected [-1, 1] range
    # UNet output is usually [-1, 1], so the loader should match this for the real data.
    data_transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(), # Scales to [0, 1]
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
        # Assuming your Rectified Flow works on normalized data that is often
        # further shifted to [-1, 1] or has its own normalization scheme.
        # ADJUST THIS SECTION to match your model's exact input normalization!
        # Example of converting [0, 1] to [-1, 1] after standard normalization:
        # transforms.Lambda(lambda t: t * 2 - 1) 
    ])
    
    train_dataset = datasets.CIFAR10(
        root=DATA_ROOT,
        train=True,
        download=True,
        transform=data_transform
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False, # Shuffle is not necessary for FID calculation
        num_workers=4,
        pin_memory=True
    )
    return train_loader

# --- 2. FID Calculation Helper Functions ---

def convert_to_uint8(tensors, data_range=(-1., 1.)):
    """
    Converts a batch of float tensors (C, H, W) to uint8 tensors (0-255).
    FID metric requires uint8 input.
    """
    tensors = tensors.detach().cpu()
    
    # Reverse normalization to get back to [0, 1] (This is complex for normalized data, 
    # so we assume model output is denormalized or that the tensors are in the DATA_RANGE)
    if data_range == (-1., 1.):
        tensors = (tensors + 1.0) / 2.0 # (-1, 1) -> (0, 1)
    elif data_range == (0., 1.):
        pass # Already in [0, 1] range
    else:
        # If your data is normalized with CIFAR mean/std, this step is tricky
        # You would need to denormalize first:
        # for t, m, s in zip(tensors, CIFAR_MEAN, CIFAR_STD): t.mul_(s).add_(m)
        # This implementation bypasses explicit denormalization and assumes the 
        # range is approximately within the [-1, 1] box for simplicity.
        pass
        
    # Scale from [0, 1] to [0, 255] and cast to uint8
    tensors = tensors.clamp(0, 1) * 255.0
    return tensors.to(torch.uint8)

def calculate_fid_for_samples(generated_samples, dataloader, device, data_range):
    """
    Calculates the FID score between generated samples and a real dataset.
    """
    # 1. Initialize the FID metric
    fid_metric = FrechetInceptionDistance(
        feature=2048, 
        normalize=False # We handle the conversion to uint8 [0, 255] ourselves
    ).to(device)
    
    # 2. Process and Update with REAL Images (from Dataloader)
    print("\n[FID] Updating with REAL images...")
    for batch in tqdm(dataloader, desc="Processing Real Data"):
        real_images = batch[0].to(device)
        # Convert the batch to the required uint8 [0, 255] format
        real_images_uint8 = convert_to_uint8(real_images, data_range=data_range).to(device)
        fid_metric.update(real_images_uint8, real=True)
        
    # 3. Process and Update with GENERATED Images (from Tensor)
    print("[FID] Updating with GENERATED images...")
    generated_uint8 = convert_to_uint8(generated_samples, data_range=data_range).to(device)
    
    # Update the metric in batches for memory efficiency
    for i in tqdm(range(0, generated_uint8.shape[0], FID_BATCH_SIZE), desc="Processing Generated Data"):
        batch = generated_uint8[i:i + FID_BATCH_SIZE]
        fid_metric.update(batch, real=False)

    # 4. Compute the final score
    fid_score = fid_metric.compute()
    return fid_score.item()

# --- 3. Example Execution (Replace with your actual Rectified Flow code) ---

# --- DUMMY DATA GENERATION (Replace with your actual gen_linear/gen_geometric) ---


Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /home/zeus/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 418MB/s]



[FID] Updating with REAL images...


Processing Real Data: 100%|██████████| 391/391 [01:24<00:00,  4.63it/s]


[FID] Updating with GENERATED images...


Processing Generated Data: 100%|██████████| 79/79 [00:16<00:00,  4.67it/s]



[FID] Updating with REAL images...


Processing Real Data: 100%|██████████| 391/391 [01:25<00:00,  4.57it/s]


[FID] Updating with GENERATED images...


Processing Generated Data: 100%|██████████| 79/79 [00:16<00:00,  4.68it/s]



FID Score (Linear Time Grid): 502.5355
FID Score (Geometric Time Grid): 497.9059


In [7]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchmetrics.image.fid import FrechetInceptionDistance
from tqdm import tqdm
import os
# from modules import UNet, UNet2  # Assuming these are available
# from rectified_flow import RectifiedFlow # Assuming this is available
# from utils import create_save_dir, plot_generated_images # Assuming these are available

# --- Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FID_BATCH_SIZE = 128
DATA_ROOT = './data_root'  # Centralized directory for all downloaded data
DATA_RANGE = (-1., 1.)    # Assumed range of your UNet output and dataloader output

# >>>>>>> USER MUST SET THIS VARIABLE IN THE MAIN SCRIPT <<<<<<<
# Example: data = 'CIFAR10' or data = 'MNIST'
# ------------------------------------------------------------------
# Assume 'data' is defined in the main script scope, e.g., data = 'CIFAR10'
# ------------------------------------------------------------------

# Dataset-specific configurations (consolidated logic)
if data == 'CIFAR10':
    IMG_SIZE = 32
    C_IN = 3
    # Standard CIFAR-10 normalization
    DATA_MEAN = [0.4914, 0.4822, 0.4465]
    DATA_STD = [0.2023, 0.1994, 0.2010]
elif data == 'MNIST':
    IMG_SIZE = 28
    C_IN = 1
    # Standard MNIST normalization (NOTE: MNIST is grayscale, so one value)
    DATA_MEAN = [0.1307]
    DATA_STD = [0.3081]
else:
    raise ValueError(f"Unsupported dataset: {data}. Choose 'CIFAR10' or 'MNIST'.")


# --- 1. Universal Dataloader Implementation ---

def get_train_dataloader_for_fid(dataset_name, batch_size, img_size, mean, std):
    """
    Loads the training set for a specified dataset (CIFAR10 or MNIST) 
    and applies transforms to match the model's [-1, 1] range.
    """
    
    # 1. Define Transforms: Resize, ToTensor, Normalize
    # NOTE: The initial ToTensor makes C=1 for MNIST.
    data_transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(), # Scales to [0, 1]
        transforms.Normalize(mean, std),
        # Convert the resulting tensor from its normalized distribution to [-1, 1] range
        # This is a common practice for VAEs/Diffusion/Flow models.
        transforms.Lambda(lambda t: t * 2 - 1) 
    ])
    
    # 2. Select Dataset Class
    if dataset_name == 'CIFAR10':
        dataset_class = datasets.CIFAR10
    elif dataset_name == 'MNIST':
        dataset_class = datasets.MNIST
    else:
        raise ValueError("Invalid dataset name provided for dataloader.")

    train_dataset = dataset_class(
        root=DATA_ROOT,
        train=True,
        download=True,
        transform=data_transform
    )
    
    # If using MNIST for FID, the InceptionV3 model requires 3 channels (RGB).
    # The dataloader must be modified to duplicate the single channel (C=1) three times.
    if dataset_name == 'MNIST':
        # Wrap the dataset to perform the channel expansion (1 to 3)
        class GrayToRGBDataset(torch.utils.data.Dataset):
            def __init__(self, dataset):
                self.dataset = dataset
            def __len__(self):
                return len(self.dataset)
            def __getitem__(self, idx):
                img, label = self.dataset[idx]
                # img is (1, H, W) float. Duplicate it along the channel dimension.
                rgb_img = img.repeat(3, 1, 1) 
                return rgb_img, label
        
        train_dataset = GrayToRGBDataset(train_dataset)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False, 
        num_workers=4,
        pin_memory=True
    )
    return train_loader

# --- 2. FID Calculation Helper Functions (No changes needed here) ---

def convert_to_uint8(tensors, data_range=(-1., 1.)):
    """
    Converts a batch of float tensors (C, H, W) in data_range to uint8 tensors (0-255).
    FID metric requires uint8 input.
    """
    tensors = tensors.detach().cpu()
    
    # Scale from data_range to [0, 1]
    if data_range == (-1., 1.):
        tensors = (tensors + 1.0) / 2.0 # (-1, 1) -> (0, 1)
    elif data_range == (0., 1.):
        pass
    else:
        # Complex denormalization needed if standard mean/std was used
        pass
        
    # Scale from [0, 1] to [0, 255] and cast to uint8
    tensors = tensors.clamp(0, 1) * 255.0
    return tensors.to(torch.uint8)

def calculate_fid_for_samples(generated_samples, dataloader, device, data_range):
    """
    Calculates the FID score between generated samples and a real dataset.
    """
    # 1. Initialize the FID metric
    # InceptionV3 feature dimension
    fid_metric = FrechetInceptionDistance(
        feature=2048, 
        normalize=False 
    ).to(device)
    
    # 2. Process and Update with REAL Images
    print("\n[FID] Updating with REAL images...")
    for batch in tqdm(dataloader, desc="Processing Real Data"):
        real_images = batch[0].to(device)
        real_images_uint8 = convert_to_uint8(real_images, data_range=data_range).to(device)
        fid_metric.update(real_images_uint8, real=True)
        
    # 3. Process and Update with GENERATED Images
    print("[FID] Updating with GENERATED images...")
    # === FIX: Ensure Generated Samples Have 3 Channels ===
    if generated_samples.shape[1] == 1:
        print("-> Duplicating generated 1-channel data to 3 channels for FID calculation.")
        # Repeat along the channel dimension (dim=1)
        generated_samples = generated_samples.repeat(1, 3, 1, 1) 
    elif generated_samples.shape[1] != 3:
        raise ValueError(f"Generated samples have an unexpected channel count: {generated_samples.shape[1]}")
    # ====================================================

    # Convert the now 3-channel data to uint8
    generated_uint8 = convert_to_uint8(generated_samples, data_range=data_range).to(device)
    
    
    for i in tqdm(range(0, generated_uint8.shape[0], FID_BATCH_SIZE), desc="Processing Generated Data"):
        batch = generated_uint8[i:i + FID_BATCH_SIZE]
        # NOTE: For MNIST, generated samples must also be C=3, which should be handled
        # by your model's generation process if it's consistent with the dataloader.
        fid_metric.update(batch, real=False)

    # 4. Compute the final score
    fid_score = fid_metric.compute()
    return fid_score.item()

# Example: How to call the new dataloader function 
# train_loader = get_train_dataloader_for_fid(DATASET_NAME, FID_BATCH_SIZE, IMG_SIZE, DATA_MEAN, DATA_STD)
# print(f"Loaded {DATASET_NAME} with batch shape: {next(iter(train_loader))[0].shape}")

In [8]:
# Assuming your samples are (N, C, H, W) float tensors in the DATA_RANGE
N_SAMPLES = 10000 
gen_linear = loaded_linear
gen_geometric = loaded_geometric
data = "MNIST"
# --- END DUMMY DATA ---

# Load CIFAR-10 dataloader
# NOTE: The dataloader must be re-initialized before each FID calculation
# to ensure the same set of real stats is compared against both generated sets.

# 1. FID Calculation for Linear Samples
train_loader = get_train_dataloader_for_fid(data, 128, IMG_SIZE, DATA_MEAN, DATA_STD)
#get_cifar10_train_dataloader(FID_BATCH_SIZE, IMG_SIZE)


fid_linear = calculate_fid_for_samples(
    generated_samples=gen_linear,
    dataloader=train_loader,
    device=device,
    data_range=DATA_RANGE
)

# 2. FID Calculation for Geometric Samples

fid_geometric = calculate_fid_for_samples(
    generated_samples=gen_geometric,
    dataloader=train_loader,
    device=device,
    data_range=DATA_RANGE
)

print("\n==================================")
print(f"FID Score (Linear Time Grid): {fid_linear:.4f}")
print(f"FID Score (Geometric Time Grid): {fid_geometric:.4f}")
print("==================================")


[FID] Updating with REAL images...


Processing Real Data: 100%|██████████| 469/469 [00:32<00:00, 14.38it/s]


[FID] Updating with GENERATED images...
-> Duplicating generated 1-channel data to 3 channels for FID calculation.


Processing Generated Data: 100%|██████████| 8/8 [00:00<00:00, 19.66it/s]



[FID] Updating with REAL images...


Processing Real Data: 100%|██████████| 469/469 [00:32<00:00, 14.44it/s]


[FID] Updating with GENERATED images...
-> Duplicating generated 1-channel data to 3 channels for FID calculation.


Processing Generated Data: 100%|██████████| 8/8 [00:00<00:00, 19.83it/s]



FID Score (Linear Time Grid): 63.5861
FID Score (Geometric Time Grid): 63.5183
